In [7]:
import pandas as pd

# Load the prepared dataset and normalize the time column.
df = pd.read_csv("../data/gold_2004-2022.csv")
df["Date"] = pd.to_datetime(df["Date"], utc=True, errors="coerce")
df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)


df['month'] = df['Date'].dt.month
df['week'] = df['Date'].dt.day_of_week
df['day'] = df['Date'].dt.day
df['hour'] = df['Date'].dt.hour
df['minute'] = df['Date'].dt.minute

df.drop(["Date"],axis=1, inplace=True)


In [8]:
# Add RSI feature

delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(window=14, min_periods=14).mean()
avg_loss = loss.rolling(window=14, min_periods=14).mean()

rs = avg_gain / avg_loss
df["rsi"] = 100 - (100 / (1 + rs))
df.loc[avg_loss == 0, "rsi"] = 100

df.dropna(subset=["rsi"], inplace=True)
df.reset_index(drop=True, inplace=True)

df["rsi"] = df["rsi"].round(1)

In [9]:
import numpy as np
# Add Ema 20 & 50 and Trend
df["EMA20"] = df["Close"].ewm(span=20).mean()
df["EMA50"] = df["Close"].ewm(span=50).mean()

df["Trend"] = np.where(df["EMA20"] > df["EMA50"], 1, 0)

In [10]:
# Add Target / y
df["target"] = (df["Close"].shift(-1) - df["Close"]) / df["Close"] * 100

In [11]:
df.head()

,Open,High,Low,Close,Volume,month,week,day,hour,minute,rsi,EMA20,EMA50,Trend,target
0,383.6,383.6,383.6,383.6,1.0,6,4,11,7,42,45.5,383.600000,383.600000,0,0.000000
1,383.6,383.6,383.6,383.6,1.0,6,4,11,7,43,45.5,383.600000,383.600000,0,0.052138
2,383.8,383.8,383.8,383.8,1.0,6,4,11,7,44,50.0,383.673439,383.669351,1,-0.130276
3,383.3,383.3,383.3,383.3,1.0,6,4,11,7,45,38.6,383.565633,383.571400,0,0.000000
4,383.3,383.3,383.3,383.3,1.0,6,4,11,7,46,43.6,383.501378,383.512693,0,0.052178


In [16]:
df.isna().sum()

Open      0
High      0
Low       0
Close     0
Volume    0
month     0
week      0
day       0
hour      0
minute    0
rsi       0
EMA20     0
EMA50     0
Trend     0
target    1
dtype: int64

In [17]:
df.dropna(inplace=True)

In [12]:
len(df)

5740272

### Splitting Dataset

In [18]:
from sklearn.model_selection import train_test_split

X = df.drop(["target"], axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Model Hyperparameter Tuning

In [19]:
import xgboost as xgb

model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.01,
    subsample=0.7,
    colsample_bytree=0.6,
    min_child_weight=10,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

## Training & Model Eval

In [20]:
model.fit(X_train, y_train)


import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Train rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE: {mae:.6f}")
print(f"R2: {r2:.6f}")

Train rows: 4592216
Test rows: 1148055
RMSE: 0.033260
MAE: 0.019436
R2: 0.003871


In [ ]:
model.save_model("xgboost_xauusd.json")